# Preprocessing ablation — does normalizing Sicilian help NLLB?

Replicates Wdowiak's text preprocessing and tests whether it helps the **pretrained** NLLB
(whose own tokenizer was trained on un-normalized web Sicilian). Three arms, scn→en only:
- **raw**: original text
- **std**: standardization spelling rules only (accents/contractions kept)
- **full**: std + uncontraction + ASCII-fold (his full preprocessing)

Each arm: fresh NLLB-1.3B + LoRA, 1 epoch on its train.scn→train.en, evaluated on its
test.scn→**test.en** (the English references are identical across arms → directly
comparable). ~1–1.5h per arm on a T4. Generated by `experiments/dataset/normalize_scn.py`.

In [1]:
!pip -q install transformers datasets peft sentencepiece sacrebleu accelerate
!pip -q uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.7 MB/s eta 0:00:00


In [2]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/SicilianNMT-colab'
except Exception:
    OUT = 'sicilian-nllb-out'
os.makedirs(OUT, exist_ok=True)
print('using', OUT)

Mounted at /content/drive
using /content/drive/MyDrive/SicilianNMT-colab


In [3]:
def read(p): return open(p, encoding='utf-8').read().splitlines()
DATA = f'{OUT}/data'
base = DATA if os.path.exists(f'{DATA}/train.scn') else '.'
print('reading data from', base)
en = {s: read(f'{base}/{s}.en') for s in ('train', 'test')}     # shared English
# scn per arm: raw -> {s}.scn, std -> {s}.std.scn, full -> {s}.full.scn
def scn(split, arm):
    suf = '' if arm == 'raw' else f'.{arm}'
    return read(f'{base}/{split}{suf}.scn')
ARMS = ['raw', 'std', 'full']
for a in ARMS:
    print(a, '| train', len(scn('train', a)), '| test', len(scn('test', a)))

reading data from /content/drive/MyDrive/SicilianNMT-colab/data
raw | train 27386 | test 1000
std | train 27386 | test 1000
full | train 27386 | test 1000


In [4]:
import os, gc, json, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
from sacrebleu.metrics import BLEU, CHRF

MODEL = 'facebook/nllb-200-1.3B'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL)
results = {}

def translate(model, texts, bs=8, max_len=160):
    tok.src_lang = 'scn_Latn'
    tgt_id = tok.convert_tokens_to_ids('eng_Latn')
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=tgt_id, max_length=max_len, num_beams=5)
        out += tok.batch_decode(gen, skip_special_tokens=True)
        print(f'  {min(i+bs, len(texts))}/{len(texts)}', end='\r')
    print(); torch.cuda.empty_cache()
    return out

def train_eval(arm):
    for v in ('model', 'ft'):
        if v in globals():
            del globals()[v]
    gc.collect(); torch.cuda.empty_cache()
    base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, torch_dtype=dtype, low_cpu_mem_usage=True).to(device)
    ft = get_peft_model(base_model, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, bias='none',
                        target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'], task_type='SEQ_2_SEQ_LM'))
    for p in ft.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    ft.config.use_cache = False
    ft.enable_input_require_grads()
    tok.src_lang, tok.tgt_lang = 'scn_Latn', 'eng_Latn'
    ds = Dataset.from_dict({'src': scn('train', arm), 'tgt': en['train']}).map(
        lambda b: tok(b['src'], text_target=b['tgt'], max_length=128, truncation=True),
        batched=True, remove_columns=['src', 'tgt']).shuffle(seed=13)
    args = Seq2SeqTrainingArguments(
        output_dir=f'{OUT}/trainer-abl-{arm}', num_train_epochs=1,
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
        learning_rate=2e-4, fp16=True, logging_steps=100, save_strategy='no', report_to=[])
    Seq2SeqTrainer(model=ft, args=args, train_dataset=ds,
                   data_collator=DataCollatorForSeq2Seq(tok, model=ft)).train()
    m = ft.eval(); m.config.use_cache = True
    hyp = translate(m, scn('test', arm))
    b = BLEU().corpus_score(hyp, [en['test']]).score
    c = CHRF().corpus_score(hyp, [en['test']]).score
    results[arm] = {'bleu': round(b, 2), 'chrf': round(c, 2)}
    json.dump(results, open(f'{OUT}/results_ablation.json', 'w'), indent=2)
    print(f'== {arm}:  scn->en  BLEU={b:.2f}  chrF={c:.2f} ==')
print('ready on', device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

ready on cuda


In [5]:
for arm in ARMS:
    print(f'\n######## arm: {arm} ########')
    train_eval(arm)
print('\nFINAL (scn->en, same English references):')
print(json.dumps(results, indent=2))


######## arm: raw ########


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

...

  1000/1000
== std:  scn->en  BLEU=30.98  chrF=56.63 ==

######## arm: full ########


Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

Map:   0%|          | 0/27386 [00:00<?, ? examples/s]

Step,Training Loss
100,7.551444
200,6.809790
300,6.595637
400,6.720934
500,6.628334
600,6.621288
700,6.534139
800,6.401097
900,6.383447
1000,6.643298


  1000/1000
== full:  scn->en  BLEU=30.98  chrF=56.50 ==

FINAL (scn->en, same English references):
{
  "raw": {
    "bleu": 30.6,
    "chrf": 56.31
  },
  "std": {
    "bleu": 30.98,
    "chrf": 56.63
  },
  "full": {
    "bleu": 30.98,
    "chrf": 56.5
  }
}
